# NeuroXNet reproducibility run (Colab)
Runs scripts 00-10 in order. Use a GPU runtime. Mount Drive to keep `predictions/`, `checkpoints/` and `results/` between sessions.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
%cd /content/drive/MyDrive
!git clone <repository-url> neuroxnet-repro || echo 'already cloned'
%cd neuroxnet-repro
!pip -q install -r environment/requirements.txt
!pytest -q tests/

In [ ]:
!bash scripts/00_download_cq500.sh

Place `data/labels/cq500_severity_labels.csv` (derived labels) before the next cell. `--make-splits` is run once only; the split files are then committed.

In [ ]:
!python scripts/01_preprocess.py --config configs/neuroxnet.yaml --make-splits

In [ ]:
!bash scripts/02_train_main.sh

In [ ]:
!bash scripts/03_train_baselines.sh

In [ ]:
!bash scripts/04_train_ablation.sh

In [ ]:
!bash scripts/05_cross_validate.sh

External cohort: run `src/neuroxnet/inference/predict.py` at the institution on the de-identified preprocessed studies and copy the resulting CSV (with `vendor`, `slice_thickness_group`, `age_group` columns) to `data/external/external_predictions_deidentified.csv`.

In [ ]:
!bash scripts/06_predict_all.sh
!python scripts/07_evaluate_all.py
!python scripts/08_gradcam_eval.py
!python scripts/09_benchmark_latency.py
!python scripts/10_make_tables_figures.py
!cat results/checks/consistency_report.txt | tail -3

In [ ]:
# bundle everything the manuscript needs
!zip -r neuroxnet_results_for_paper.zip results predictions checkpoints/SHA256SUMS.txt logs/*.json logs/*.csv -x '*.pt'